# corrected nested data-size ablation

Run this notebook in a fresh Kaggle GPU session with Internet enabled. It trains the same SmolLM3 QLoRA recipe on nested 40, 80, and 160-row prefixes for seeds 42, 43, and 44, then scores every run on the frozen 50-row validation split. The 40-row prefix is contained in the 80-row prefix for each seed.

This is a supplemental ablation only. It does not load the internal test or external-transfer files, and it does not change Evaluation v2 or the frozen final scores.

In [ ]:
!pip -q install 'transformers==5.10.1' 'peft==0.20.0' 'trl==0.29.0' bitsandbytes accelerate datasets 'mlflow==3.15.1' scikit-learn

import hashlib
import json
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

REPO_REF = '6c1c733'
RAW_BASE = f'https://raw.githubusercontent.com/goyashek/civic-grievance-structurer/{REPO_REF}'
ROOT = Path('/kaggle/working/civicstruct')
ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_FILES = (
    'src/evaluate.py',
    'src/schema.py',
    'data/surface_variants.jsonl',
    'data/public_training_examples.jsonl',
    'data/dataset_manifest.json',
    'data/validation_results/qlora_validation_predictions.json',
)

def fetch(relative_path):
    destination = ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(f'{RAW_BASE}/{relative_path}', timeout=60) as response:
            destination.write_bytes(response.read())
    except URLError as exc:
        raise RuntimeError('Turn on Kaggle Internet to fetch the frozen project files.') from exc

for relative_path in TRAIN_FILES:
    fetch(relative_path)
print({'source': RAW_BASE, 'fetched': list(TRAIN_FILES), 'test_loaded': False})

In [ ]:
import gc
import enum
import math
import os
import platform
import random
import shutil
import sys
import time
from collections import defaultdict
from importlib.metadata import version
from statistics import mean, stdev

sys.path.insert(0, str(ROOT))
import mlflow
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from src.evaluate import evaluate_outputs
from src.schema import validate_gold

SEED = 42
RUN_SEEDS = (42, 43, 44)
ROW_COUNTS = (40, 80, 160)
MODEL_NAME = 'HuggingFaceTB/SmolLM3-3B'
MODEL_REVISION = 'a07cc9a04f16550a088caea529712d1d335b0ac1'
REVISION_EPOCHS = 2
MAX_NEW_TOKENS = 256
MAX_LENGTH = 768
DEMO_COUNT = 3
BATCH_SIZE = 4
MODEL_DTYPE = torch.float16
OUTPUT = Path('/kaggle/working/civicstruct_nested_ablation_output')
OUTPUT.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select a Kaggle GPU runtime before running this notebook.'
torch.cuda.manual_seed_all(SEED)
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
mlflow.set_tracking_uri((OUTPUT / 'mlruns').as_uri())
mlflow.set_experiment('civicstruct-ablation-nested')

def json_default(value):
    if isinstance(value, set):
        return sorted(value, key=str)
    if isinstance(value, (Path, os.PathLike)):
        return os.fspath(value)
    if isinstance(value, torch.dtype):
        return str(value)
    if isinstance(value, enum.Enum):
        return value.value
    if hasattr(value, 'item'):
        try:
            return value.item()
        except (TypeError, ValueError):
            pass
    if hasattr(value, 'tolist'):
        return value.tolist()
    return str(value)

def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=json_default), encoding='utf-8')

def load_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print({'device': torch.cuda.get_device_name(0), 'python': platform.python_version(), 'torch': torch.__version__})

In [ ]:
manifest = json.loads((ROOT / 'data/dataset_manifest.json').read_text(encoding='utf-8'))
for relative_path in ('data/surface_variants.jsonl', 'data/public_training_examples.jsonl'):
    assert sha256(ROOT / relative_path) == manifest['sha256'][relative_path], relative_path

surfaces = load_jsonl(ROOT / 'data/surface_variants.jsonl')
public_training = load_jsonl(ROOT / 'data/public_training_examples.jsonl')
controlled_training = [row for row in surfaces if row['split'] == 'train']
validation = [row for row in surfaces if row['split'] == 'validation']
training = [dict(row) for row in controlled_training] + [dict(row) for row in public_training]
for row in training + validation:
    row.setdefault('surface_id', row['case_id'])
assert len(controlled_training) == 120
assert len(public_training) == 40
assert len(training) == 160
assert len(validation) == 50
assert len({row['surface_id'] for row in validation}) == len(validation)
assert len({row['case_id'] for row in training}) == len(training)
assert not {row['case_id'] for row in training} & {row['case_id'] for row in validation}
print({'training': len(training), 'validation': len(validation), 'test_loaded': False, 'dataset_version': manifest['dataset_version']})

In [ ]:
DOMAINS = ['public_transport', 'water_supply', 'sanitation_and_waste', 'roads_and_streetlights', 'electricity', 'welfare_or_document_service', 'other']
ISSUES = ['delay_or_non_arrival', 'service_outage_or_non_delivery', 'damaged_infrastructure', 'overcharging_or_payment_problem', 'record_or_document_error', 'staff_conduct', 'safety_or_health_hazard', 'other']
URGENCY = ['routine', 'time_sensitive', 'safety_critical']
MISSING = ['exact_location', 'date_or_time', 'service_identifier', 'transaction_or_reference_id', 'amount', 'supporting_evidence', 'affected_person_or_group', 'none']
SYSTEM_PROMPT = (
    'Structure one public-service complaint as exactly one JSON object. Use these fields in this order: '
    'service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, urgency, missing_information, formal_summary. '
    f'Allowed service_domain values: {DOMAINS}. Allowed issue_type values: {ISSUES}. '
    f'Allowed urgency values: {URGENCY}. Allowed missing_information values: {MISSING}. '
    'Use null for absent scalar facts. Missing information must be a non-empty ordered list. Use the none label only when no important detail is missing. '
    'Do not guess facts. The formal summary must be one neutral sentence. Return no reasoning, markdown, or commentary.'
)

def messages_for(complaint, demos=()):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for demo in demos:
        messages.extend([
            {'role': 'user', 'content': demo['complaint']},
            {'role': 'assistant', 'content': json.dumps(demo['gold'], ensure_ascii=False, separators=(',', ':'))},
        ])
    messages.append({'role': 'user', 'content': complaint})
    return messages

def train_records(rows):
    return [
        {'prompt': messages_for(row['complaint']), 'completion': [{'role': 'assistant', 'content': json.dumps(row['gold'], ensure_ascii=False, separators=(',', ':'))}]}
        for row in rows
    ]

STATIC_IDS = ('canonical-001', 'canonical-020', 'canonical-030')
training_by_id = {row['case_id']: row for row in training}
static_demos = [training_by_id[case_id] for case_id in STATIC_IDS]
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=1)
training_matrix = vectorizer.fit_transform([row['complaint'] for row in training])

def retrieve(complaint):
    scores = (training_matrix @ vectorizer.transform([complaint]).T).toarray().ravel()
    ranked = scores.argsort()[::-1]
    selected = [training[index] for index in ranked[:DEMO_COUNT]]
    assert len({row['case_id'] for row in selected}) == DEMO_COUNT
    return selected

def retrieved_lookup(rows):
    selected = {row['surface_id']: retrieve(row['complaint']) for row in rows}
    assert all(all(demo['split'] == 'train' for demo in demos) for demos in selected.values())
    return selected

retrieved_validation = retrieved_lookup(validation)
print({'static_demo_ids': list(STATIC_IDS), 'retrieval_training_rows': len(training), 'validation_retrieval_ready': len(retrieved_validation)})

In [ ]:
def rule_output(complaint):
    text = complaint.casefold()
    if any(word in text for word in ('water', 'tap ', 'tanker')):
        domain = 'water_supply'
    elif any(word in text for word in ('garbage', 'waste', 'sewage', 'sanitation', 'bin ', 'sweeper')):
        domain = 'sanitation_and_waste'
    elif any(word in text for word in ('electric', 'power', 'feeder', 'transformer')):
        domain = 'electricity'
    elif any(word in text for word in ('road', 'streetlight', 'street light', 'pothole', 'signal', 'sidewalk', 'pavement', 'parking meter')):
        domain = 'roads_and_streetlights'
    elif any(word in text for word in ('pension', 'certificate', 'benefit', 'welfare', 'application', 'document')):
        domain = 'welfare_or_document_service'
    elif any(word in text for word in ('bus', 'train', 'station', 'platform', 'ticket', 'route ')):
        domain = 'public_transport'
    else:
        domain = 'other'
    if any(word in text for word in ('charged', 'bill ', 'fee ', 'payment')):
        issue = 'overcharging_or_payment_problem'
    elif any(word in text for word in ('wrong', 'incorrect', 'marked completed', 'record ')):
        issue = 'record_or_document_error'
    elif any(word in text for word in ('shouted', 'insulted', 'rude', 'refused', 'mocked', 'ignored')):
        issue = 'staff_conduct'
    elif any(word in text for word in ('live wire', 'sparking', 'unsafe', 'hazard', 'collision', 'needles', 'sharp edges')):
        issue = 'safety_or_health_hazard'
    elif any(word in text for word in ('did not arrive', 'never arrived', 'never came', 'did not come', 'scheduled for', 'was due')):
        issue = 'delay_or_non_arrival'
    elif any(word in text for word in ('broken', 'cracked', 'damaged', 'pothole', 'leaking', 'raised sidewalk', 'leaning')):
        issue = 'damaged_infrastructure'
    elif any(word in text for word in ('no water', 'no power', 'unavailable', 'not working', 'has not worked', 'error for', 'no collection')):
        issue = 'service_outage_or_non_delivery'
    else:
        issue = 'other'
    urgency = 'safety_critical' if issue == 'safety_or_health_hazard' else ('time_sensitive' if any(word in text for word in ('for two days', 'for three days', 'for four days', 'for five days', 'for six days', 'for a week', 'since monday', 'since friday', 'since sunday')) else 'routine')
    return {'service_domain': domain, 'issue_type': issue, 'location': None, 'event_date_or_time': None, 'amount_inr': None, 'service_identifier': None, 'urgency': urgency, 'missing_information': ['exact_location'], 'formal_summary': complaint.strip()}

quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=MODEL_DTYPE, bnb_4bit_use_double_quant=True)

def load_base():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, quantization_config=quantization_config, dtype=MODEL_DTYPE, device_map='auto')
    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id
    model.config.use_cache = True
    return tokenizer, model

def encoded_batch(tokenizer, message_batch):
    kwargs = dict(tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt', padding=True, truncation=True, max_length=2048, enable_thinking=False)
    try:
        return tokenizer.apply_chat_template(message_batch, **kwargs)
    except TypeError:
        kwargs.pop('enable_thinking')
        return tokenizer.apply_chat_template(message_batch, **kwargs)

def generate_rows(tokenizer, model, rows, demo_lookup=None):
    outputs = []
    model.eval()
    for start in range(0, len(rows), BATCH_SIZE):
        batch = rows[start:start + BATCH_SIZE]
        messages = [messages_for(row['complaint'], () if demo_lookup is None else demo_lookup[row['surface_id']]) for row in batch]
        inputs = encoded_batch(tokenizer, messages).to(next(model.parameters()).device)
        prompt_lengths = inputs['attention_mask'].sum(dim=1).tolist()
        padded_length = inputs['input_ids'].shape[1]
        torch.cuda.synchronize()
        started = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**inputs, do_sample=False, max_new_tokens=MAX_NEW_TOKENS, use_cache=True, pad_token_id=tokenizer.pad_token_id)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - started
        responses = tokenizer.batch_decode(generated[:, padded_length:], skip_special_tokens=True)
        outputs.extend([{'surface_id': row['surface_id'], 'case_id': row['case_id'], 'response': response.strip(), 'latency_seconds': elapsed / len(batch), 'prompt_tokens': int(prompt_tokens)} for row, response, prompt_tokens in zip(batch, responses, prompt_lengths)])
    return outputs

def prepare_trainable_parameters(model):
    if hasattr(model, 'enable_input_require_grads'):
        model.enable_input_require_grads()
    for parameter in model.parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            parameter.data = parameter.data.float()
    assert all(parameter.dtype == torch.float32 for parameter in model.parameters() if parameter.requires_grad)

In [ ]:
def score_record(method, rows, outputs, *, model_name=MODEL_NAME, model_revision=MODEL_REVISION, demos=0, extra=None):
    scores = evaluate_outputs([row['gold'] for row in rows], [item['response'] for item in outputs])
    record = {'method': method, 'model_name': model_name, 'model_revision': model_revision, 'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS}, 'mean_latency_seconds': sum(item['latency_seconds'] for item in outputs) / len(outputs), 'mean_prompt_tokens': sum(item['prompt_tokens'] for item in outputs) / len(outputs), 'demonstration_count': demos, 'scores': scores, 'outputs': outputs}
    if extra:
        record.update(extra)
    with mlflow.start_run(run_name=method) as run:
        record['mlflow_run_id'] = run.info.run_id
        mlflow.log_params({'method': method, 'model_name': model_name or 'none', 'model_revision': model_revision or 'none', 'demonstration_count': demos})
        mlflow.log_metrics({'schema_validity_rate': scores['strict']['schema_validity_rate'], 'mean_latency_seconds': record['mean_latency_seconds'], 'mean_prompt_tokens': record['mean_prompt_tokens']})
        mlflow.log_text(json.dumps(record, indent=2, ensure_ascii=False, default=json_default), 'results.json')
    return record

def train_one(rows, epochs, run_name, save_dir=None, seed=SEED, subset_name=None, case_ids=None):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    tokenizer, base = load_base()
    base.config.use_cache = False
    torch.cuda.reset_peak_memory_stats()
    lora_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules='all-linear')
    args = SFTConfig(output_dir=f'/kaggle/working/{run_name}_checkpoints', num_train_epochs=epochs, per_device_train_batch_size=1, gradient_accumulation_steps=8, learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine', logging_steps=1, save_strategy='no', report_to='none', fp16=False, bf16=False, max_length=MAX_LENGTH, completion_only_loss=True, gradient_checkpointing=True, seed=seed)
    trainer = SFTTrainer(model=base, args=args, train_dataset=Dataset.from_list(train_records(rows)), processing_class=tokenizer, peft_config=lora_config)
    prepare_trainable_parameters(trainer.model)
    trainable = sum(parameter.numel() for parameter in trainer.model.parameters() if parameter.requires_grad)
    total = sum(parameter.numel() for parameter in trainer.model.parameters())
    assert 0 < trainable < total
    started = time.perf_counter()
    train_result = trainer.train()
    training_seconds = time.perf_counter() - started
    losses = [item['loss'] for item in trainer.state.log_history if 'loss' in item]
    assert losses and all(math.isfinite(loss) for loss in losses)
    if save_dir is not None:
        save_dir = Path(save_dir)
        if save_dir.exists():
            shutil.rmtree(save_dir)
        trainer.save_model(save_dir)
        tokenizer.save_pretrained(save_dir)
    metadata = {'run_name': run_name, 'seed': seed, 'subset_name': subset_name, 'case_ids': case_ids or [], 'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_rows': len(rows), 'dataset_version': manifest['dataset_version'], 'epochs': epochs, 'configuration': args.to_dict(), 'lora': lora_config.to_dict(), 'losses': losses, 'train_metrics': train_result.metrics, 'training_seconds': training_seconds, 'peak_gpu_memory_mb': torch.cuda.max_memory_allocated() / 1024**2, 'device': torch.cuda.get_device_name(0), 'trainable_parameters': trainable, 'total_parameters': total, 'packages': {name: version(name) for name in ('torch', 'transformers', 'peft', 'trl', 'bitsandbytes', 'datasets', 'mlflow')}}
    with mlflow.start_run(run_name=run_name) as run:
        metadata['mlflow_run_id'] = run.info.run_id
        mlflow.log_params({'model_name': MODEL_NAME, 'model_revision': MODEL_REVISION, 'dataset_rows': len(rows), 'epochs': epochs, 'seed': seed, 'subset_name': subset_name or 'unspecified', 'lora_r': 16, 'learning_rate': 2e-4})
        mlflow.log_metrics({'train_loss': train_result.metrics['train_loss'], 'training_seconds': training_seconds, 'peak_gpu_memory_mb': metadata['peak_gpu_memory_mb']})
        mlflow.log_text(json.dumps(metadata, indent=2, default=json_default), 'training_metadata.json')
        if save_dir is not None:
            mlflow.log_artifacts(str(save_dir), artifact_path='adapter')
    return {'trainer': trainer, 'tokenizer': tokenizer, 'metadata': metadata, 'adapter_dir': save_dir}

def validation_record(model, tokenizer, name, extra=None):
    outputs = generate_rows(tokenizer, model, validation)
    return score_record(name, validation, outputs, extra=extra)

## nested prefixes and repeated seeds

The ordering is stratified by service domain with shuffled rows inside each domain. The first 40 rows cover all service domains. Every larger prefix reuses the earlier rows, so training size is the only changing data dimension within a seed. Each run saves its adapter and logs its seed, subset, metrics, duration, and peak GPU memory to MLflow.

In [ ]:
def stratified_order(rows, seed):
    buckets = {domain: [] for domain in DOMAINS}
    for row in rows:
        buckets[row['gold']['service_domain']].append(row)
    rng = random.Random(seed)
    for bucket in buckets.values():
        rng.shuffle(bucket)
    ordered = []
    while any(buckets.values()):
        for domain in DOMAINS:
            if buckets[domain]:
                ordered.append(buckets[domain].pop())
    assert len(ordered) == len(rows)
    assert len({row['case_id'] for row in ordered}) == len(rows)
    return ordered

def metric_value(scores, path):
    value = scores
    for key in path:
        value = value[key]
    return float(value)

METRICS = {
    'schema_validity': ('strict', 'schema_validity_rate'),
    'domain_macro_f1': ('strict', 'end_to_end_field_metrics', 'service_domain_macro_f1'),
    'issue_macro_f1': ('strict', 'end_to_end_field_metrics', 'issue_type_macro_f1'),
    'urgency_macro_f1': ('strict', 'end_to_end_field_metrics', 'urgency_macro_f1'),
    'missing_information_macro_f1': ('strict', 'end_to_end_field_metrics', 'missing_information_macro_f1'),
}

runs = []
nested_orders = {}
for seed in RUN_SEEDS:
    ordered = stratified_order(training, seed)
    nested_orders[str(seed)] = [row['case_id'] for row in ordered]
    previous_ids = set()
    for row_count in ROW_COUNTS:
        subset = ordered[:row_count]
        case_ids = [row['case_id'] for row in subset]
        current_ids = set(case_ids)
        assert previous_ids <= current_ids
        previous_ids = current_ids
        assert set(row['gold']['service_domain'] for row in subset) == set(DOMAINS)
        run_name = f'nested-ablation-{row_count}-rows-seed-{seed}'
        adapter_dir = OUTPUT / 'adapters' / run_name
        trained = train_one(
            subset, REVISION_EPOCHS, run_name, save_dir=adapter_dir, seed=seed,
            subset_name=f'{row_count}_rows', case_ids=case_ids,
        )
        validation_result = validation_record(
            trained['trainer'].model, trained['tokenizer'], f'{run_name}-validation',
            extra={'seed': seed, 'subset_rows': row_count, 'subset_case_ids': case_ids},
        )
        runs.append({
            'seed': seed,
            'rows': row_count,
            'case_ids': case_ids,
            'training': trained['metadata'],
            'validation': validation_result,
        })
        del trained['trainer'], trained['tokenizer']
        gc.collect()
        torch.cuda.empty_cache()

summary = {}
for row_count in ROW_COUNTS:
    selected = [run for run in runs if run['rows'] == row_count]
    summary[str(row_count)] = {}
    for name, path in METRICS.items():
        values = [metric_value(run['validation']['scores'], path) for run in selected]
        summary[str(row_count)][name] = {
            'values': values,
            'mean': mean(values),
            'std': stdev(values),
            'n': len(values),
        }

result = {
    'experiment_type': 'corrected_nested_data_size_ablation',
    'evaluation_version': runs[0]['validation']['scores']['evaluation_version'],
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'dataset_version': manifest['dataset_version'],
    'training_recipe': {'epochs': REVISION_EPOCHS, 'learning_rate': 2e-4, 'lora_r': 16, 'lora_alpha': 32},
    'seeds': list(RUN_SEEDS),
    'row_counts': list(ROW_COUNTS),
    'stratification': 'round_robin_service_domain',
    'nested_orders': nested_orders,
    'runs': runs,
    'summary': summary,
    'test_loaded': False,
}
save_json(OUTPUT / 'ablation_results.json', result)
save_json(OUTPUT / 'run_summary.json', {'experiment_type': result['experiment_type'], 'summary': summary, 'test_loaded': False})
archive = shutil.make_archive('/kaggle/working/civicstruct_nested_ablation', 'zip', root_dir=str(OUTPUT))
print({'archive': archive, 'runs': len(runs), 'row_counts': list(ROW_COUNTS), 'seeds': list(RUN_SEEDS), 'test_loaded': False})